In [1]:
import torch, torchaudio, random, itertools
import torch.nn as nn
import numpy as np, pandas as pd
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAMPLE_RATE, HOP_SECONDS = 44100, 0.010
HOP, N_FFT, N_MELS = int(HOP_SECONDS * SAMPLE_RATE), 2048, 128

class OnsetCRNN(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.mel   = torchaudio.transforms.MelSpectrogram(
            sample_rate=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS)
        self.to_db = torchaudio.transforms.AmplitudeToDB()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(32), nn.MaxPool2d((2,1)),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(64), nn.MaxPool2d((2,1)),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(128),
        )
        self.rnn = nn.GRU(128 * (N_MELS // 4), 128, num_layers=3, batch_first=True,
                          bidirectional=True, dropout=dropout)
        self.drop = nn.Dropout(dropout)
        self.state_head, self.onset_head, self.offset_head = nn.Linear(256, 2), nn.Linear(256, 1), nn.Linear(256, 1)
    def forward(self, x):
        s = self.cnn(self.to_db(self.mel(x)))
        B, C, M, T = s.shape
        s, _ = self.rnn(s.reshape(B, C*M, T).transpose(1, 2))
        s = self.drop(s)
        return self.state_head(s), self.onset_head(s).squeeze(-1), self.offset_head(s).squeeze(-1)

weight_files = sorted(Path("/kaggle/input").rglob("biggerCRNN_ext4_seed*.pt"))

# ---- SVNote1 files ----
SV = Path("/kaggle/input/datasets/alirahmatpour/vocalset-onset/svnote_upload")
ANNOTATORS = ["dooyong", "gwangseok", "sanghee"]

def key(name):
    """'man6_1_schoolbell.wav.csv' / 'man6_schoolbell.wav' -> 'man6_schoolbell' (drop number-only parts)."""
    stem = name.split(".")[0]
    return "_".join(p for p in stem.split("_") if not p.isdigit())

wavs = {key(w.name): w for w in sorted((SV / "audio").rglob("*.wav")) if not w.name.startswith("._")}
annots = {}
for a in ANNOTATORS:
    csvs = [c for c in (SV / "annot" / a).rglob("*.csv") if not c.name.startswith("._")]
    annots[a] = {key(c.name): c for c in csvs}

# ---- checks ----
print("device :", device, "| weights:", [w.name for w in weight_files], "(expect 3)")
print("audio files:", len(wavs))
for a in ANNOTATORS:
    missing = sorted(set(wavs) - set(annots[a]))
    extra   = sorted(set(annots[a]) - set(wavs))
    print(f"{a:10s}: {len(annots[a])} csvs | wavs without csv: {len(missing)} {missing[:5]} | csvs without wav: {len(extra)} {extra[:5]}")
common = sorted(set(wavs).intersection(*[set(annots[a]) for a in ANNOTATORS]))
print("songs with all 3 annotators:", len(common))
print("example keys:", common[:5])

device : cuda | weights: ['biggerCRNN_ext4_seed0.pt', 'biggerCRNN_ext4_seed1.pt', 'biggerCRNN_ext4_seed2.pt'] (expect 3)
audio files: 27
dooyong   : 27 csvs | wavs without csv: 0 [] | csvs without wav: 0 []
gwangseok : 27 csvs | wavs without csv: 0 [] | csvs without wav: 0 []
sanghee   : 27 csvs | wavs without csv: 0 [] | csvs without wav: 0 []
songs with all 3 annotators: 27
example keys: ['man1_butterfly', 'man1_schoolbell', 'man1_twinkle', 'man2_butterfly', 'man2_schoolbell']


In [2]:
# ---- load audio + onsets from each annotator ----
songs = []
for k in common:
    wf, sr = torchaudio.load(str(wavs[k]))
    wave = wf.mean(0)                                   # mono
    if sr != SAMPLE_RATE:
        wave = torchaudio.functional.resample(wave, sr, SAMPLE_RATE)
    on = {a: np.loadtxt(annots[a][k], delimiter=",", ndmin=2)[:, 0] for a in ANNOTATORS}
    songs.append({"key": k, "singer": k.split("_")[0], "sr": sr, "wave": wave, "onsets": on})
print("songs:", len(songs), "| sample rates:", sorted({s['sr'] for s in songs}),
      "| singers:", sorted({s['singer'] for s in songs}))
print("notes per annotator:", {a: sum(len(s['onsets'][a]) for s in songs) for a in ANNOTATORS})

# ---- scoring (same as always: peak picking, frames of 10 ms) ----
def pick_peaks(prob, thr, min_gap=3):
    peaks = []
    for i in range(len(prob)):
        if prob[i] < thr: continue
        lo, hi = max(0, i-1), min(len(prob), i+2)
        if prob[i] == prob[lo:hi].max() and (not peaks or i - peaks[-1] >= min_gap):
            peaks.append(i)
    return set(peaks)

def f1_sets(pred_list, true_list, tol):
    tp = fp = fn = 0
    for pred, t_idx in zip(pred_list, true_list):
        matched = set()
        for ti in t_idx:
            hit = next((pi for pi in pred if abs(pi - ti) <= tol and pi not in matched), None)
            if hit is not None: tp += 1; matched.add(hit)
            else: fn += 1
        fp += len(set(pred) - matched)
    p = tp/(tp+fp) if tp+fp else 0.0; r = tp/(tp+fn) if tp+fn else 0.0
    return 2*p*r/(p+r) if p+r else 0.0

frames = {a: [set((s["onsets"][a] / HOP_SECONDS).astype(int)) for s in songs] for a in ANNOTATORS}

# ---- 1) human ceiling: annotator vs annotator ----
print("\n=== HUMAN CEILING: annotator vs annotator (onset F1) ===")
hum = []
for a, b in itertools.combinations(ANNOTATORS, 2):
    f20, f50 = f1_sets(frames[a], frames[b], 2), f1_sets(frames[a], frames[b], 5)
    hum.append((f20, f50))
    print(f"{a:10s} vs {b:10s} | ±20ms {f20:.3f} | ±50ms {f50:.3f}")
print(f"average              | ±20ms {np.mean([h[0] for h in hum]):.3f} | ±50ms {np.mean([h[1] for h in hum]):.3f}")

# ---- 2) zero-shot: VocalSet models on SVNote1 ----
rows, offsets = [], []
for w in weight_files:
    model = OnsetCRNN().to(device)
    model.load_state_dict(torch.load(w, map_location=device)); model.eval()
    with torch.no_grad():
        probs = [torch.sigmoid(model(s["wave"].view(1,1,-1).to(device))[1])[0].cpu().numpy() for s in songs]
    for a in ANNOTATORS:
        peaks08 = [pick_peaks(p, 0.8) for p in probs]
        best_thr = max([0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9],
                       key=lambda th: f1_sets([pick_peaks(p, th) for p in probs], frames[a], 2))
        peaksB = [pick_peaks(p, best_thr) for p in probs]
        rows.append({"seed": w.stem[-1], "annotator": a,
                     "F1_20": f1_sets(peaks08, frames[a], 2), "F1_50": f1_sets(peaks08, frames[a], 5),
                     "best_thr": best_thr, "bestF1_20": f1_sets(peaksB, frames[a], 2),
                     "bestF1_50": f1_sets(peaksB, frames[a], 5)})
        # timing offset: prediction - nearest true onset (within ±200 ms)
        for pred, t in zip(peaks08, frames[a]):
            t = np.array(sorted(t))
            for pi in pred:
                dd = pi - t[np.argmin(np.abs(t - pi))]
                if abs(dd) <= 20: offsets.append(dd * 10)
    del model; torch.cuda.empty_cache()

zs = pd.DataFrame(rows)
print("\n=== ZERO-SHOT: VocalSet model on SVNote1 (avg over 3 annotators; mean ± std over 3 seeds) ===")
per_seed = zs.groupby("seed")[["F1_20", "F1_50", "bestF1_20", "bestF1_50"]].mean()
for c in per_seed.columns:
    print(f"{c:10s}: {per_seed[c].mean():.3f} ± {per_seed[c].std(ddof=1):.3f}")
print("best thresholds picked:", sorted(zs.best_thr.unique()))
off = np.array(offsets)
print(f"\ntiming offset (model - human): median {np.median(off):+.0f} ms | within ±20ms: {np.mean(np.abs(off)<=20):.0%}")

songs: 27 | sample rates: [44100] | singers: ['man1', 'man2', 'man3', 'man4', 'man5', 'man6', 'woman1', 'woman2', 'woman3']
notes per annotator: {'dooyong': 1296, 'gwangseok': 1296, 'sanghee': 1296}

=== HUMAN CEILING: annotator vs annotator (onset F1) ===
dooyong    vs gwangseok  | ±20ms 0.708 | ±50ms 0.955
dooyong    vs sanghee    | ±20ms 0.684 | ±50ms 0.959
gwangseok  vs sanghee    | ±20ms 0.704 | ±50ms 0.958
average              | ±20ms 0.698 | ±50ms 0.957

=== ZERO-SHOT: VocalSet model on SVNote1 (avg over 3 annotators; mean ± std over 3 seeds) ===
F1_20     : 0.431 ± 0.026
F1_50     : 0.710 ± 0.042
bestF1_20 : 0.440 ± 0.017
bestF1_50 : 0.718 ± 0.039
best thresholds picked: [np.float64(0.6), np.float64(0.7), np.float64(0.8)]

timing offset (model - human): median -20 ms | within ±20ms: 46%


In [3]:
import torch.optim as optim

# ---- consensus labels: average of the 3 annotators, note by note ----
for s in songs:
    tabs = [np.loadtxt(annots[a][s["key"]], delimiter=",", ndmin=2) for a in ANNOTATORS]
    if len({len(t) for t in tabs}) == 1:
        avg = np.mean([t[:, :2] for t in tabs], axis=0)       # mean onset, mean offset
    else:
        avg = tabs[0][:, :2]                                   # fallback: first annotator
        print("note count differs, using", ANNOTATORS[0], "for", s["key"])
    s["cons_on"], s["cons_off"] = avg[:, 0], avg[:, 1]

# ---- split by singer: 2 men + 1 woman test, rest train ----
rng = random.Random(0)
men   = sorted({s["singer"] for s in songs if s["singer"].startswith("man")})
women = sorted({s["singer"] for s in songs if s["singer"].startswith("woman")})
rng.shuffle(men); rng.shuffle(women)
test_singers = set(men[:2] + women[:1])
test_i  = [i for i, s in enumerate(songs) if s["singer"] in test_singers]
pool_i  = [i for i, s in enumerate(songs) if s["singer"] not in test_singers]
rng.shuffle(pool_i)                                            # first N = the N training songs
print("test singers:", sorted(test_singers), "| test songs:", len(test_i), "| training pool:", len(pool_i))

# ---- training targets from consensus labels ----
SOFT_WIDTH = 2
def make_targets(s):
    nfr = 1 + s["wave"].shape[0] // HOP
    st = np.zeros(nfr, np.int64); on_t = np.zeros(nfr, np.float32); off_t = np.zeros(nfr, np.float32)
    def bump(arr, c):
        for d in range(-SOFT_WIDTH, SOFT_WIDTH + 1):
            j = c + d
            if 0 <= j < nfr: arr[j] = max(arr[j], np.exp(-(d*d) / (2 * (SOFT_WIDTH/2)**2)))
    for a, b in zip(s["cons_on"], s["cons_off"]):
        fa, fb = int(a / HOP_SECONDS), int(b / HOP_SECONDS)
        st[fa:fb] = 1; bump(on_t, fa); bump(off_t, fb)
    return torch.tensor(st), torch.tensor(on_t), torch.tensor(off_t)
sv_tg = [make_targets(s) for s in songs]

# ---- training: pretrained (VocalSet weights) or scratch (random start) ----
def train_sv(start, train_idx, seed, epochs, lr, crop=1500):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    model = OnsetCRNN().to(device)
    if start is not None:
        model.load_state_dict(torch.load(start, map_location=device))
    if not train_idx: return model
    opt = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    ce, bce = nn.CrossEntropyLoss(), nn.BCEWithLogitsLoss(pos_weight=torch.tensor(20.0, device=device))
    for ep in range(epochs):
        model.train()
        for i in random.sample(train_idx, len(train_idx)):
            st, on, off = sv_tg[i]
            nfr = len(st); a = np.random.randint(0, max(1, nfr - crop)); b = min(nfr, a + crop)
            ps, po, pf = model(songs[i]["wave"][a*HOP : b*HOP].view(1, 1, -1).to(device))
            F = min(ps.shape[1], b - a)
            loss = (ce(ps[0, :F], st[a:a+F].to(device)) + bce(po[0, :F], on[a:a+F].to(device))
                    + bce(pf[0, :F], off[a:a+F].to(device)))
            opt.zero_grad(); loss.backward(); opt.step()
    return model

def evaluate_sv(model):
    """F1 on test singers, averaged over the 3 annotators."""
    model.eval()
    with torch.no_grad():
        probs = [torch.sigmoid(model(songs[i]["wave"].view(1,1,-1).to(device))[1])[0].cpu().numpy() for i in test_i]
    out = {}
    for thr_name, thr in [("", 0.8), ("best_", None)]:
        if thr is None:
            thr = max([0.3,0.4,0.5,0.6,0.7,0.8,0.9], key=lambda th: np.mean(
                [f1_sets([pick_peaks(p, th) for p in probs], [frames[a][i] for i in test_i], 5) for a in ANNOTATORS]))
        peaks = [pick_peaks(p, thr) for p in probs]
        for tol_name, tol in [("20", 2), ("50", 5)]:
            out[f"{thr_name}F1_{tol_name}"] = np.mean([f1_sets(peaks, [frames[a][i] for i in test_i], tol) for a in ANNOTATORS])
    return out

# ---- SAME training for both: 40 rounds, same step size (fair comparison) ----
SETUPS = {"pretrained": dict(epochs=40, lr=1e-3),     # starts from VocalSet weights
          "scratch":    dict(epochs=40, lr=1e-3)}     # starts empty
rows = []
for N in [0, 3, 9, len(pool_i)]:
    for k, w in enumerate(weight_files):
        for name, cfg in SETUPS.items():
            if N == 0 and name == "scratch": continue
            model = train_sv(w if name == "pretrained" else None, pool_i[:N], seed=k, **cfg)
            r = evaluate_sv(model); r.update({"N": N, "start": name, "seed": k}); rows.append(r)
            print(f"N={N:2d} | {name:10s} | seed {k} | ±20 {r['F1_20']:.3f} | ±50 {r['F1_50']:.3f}"
                  f" | best thr ±50 {r['best_F1_50']:.3f}")
            del model; torch.cuda.empty_cache()

sv_res = pd.DataFrame(rows)
sv_res.to_csv("/kaggle/working/onset_V3_svnote_zeroshot_vs_finetune.csv", index=False)
print("\n=== SVNote1 test singers — mean ± std over 3 seeds (both: 40 rounds, lr 1e-3; avg over 3 annotators) ===")
for (N, start), g in sv_res.groupby(["N", "start"]):
    print(f"N={N:2d} {start:10s} | ±20 {g.F1_20.mean():.3f} ± {g.F1_20.std(ddof=1):.3f}"
          f" | ±50 {g.F1_50.mean():.3f} ± {g.F1_50.std(ddof=1):.3f} | best-thr ±50 {g.best_F1_50.mean():.3f}")

test singers: ['man3', 'man5', 'woman1'] | test songs: 9 | training pool: 18
N= 0 | pretrained | seed 0 | ±20 0.402 | ±50 0.633 | best thr ±50 0.658
N= 0 | pretrained | seed 1 | ±20 0.418 | ±50 0.743 | best thr ±50 0.763
N= 0 | pretrained | seed 2 | ±20 0.444 | ±50 0.713 | best thr ±50 0.716
N= 3 | pretrained | seed 0 | ±20 0.751 | ±50 0.957 | best thr ±50 0.957
N= 3 | scratch    | seed 0 | ±20 0.654 | ±50 0.829 | best thr ±50 0.834
N= 3 | pretrained | seed 1 | ±20 0.708 | ±50 0.929 | best thr ±50 0.932
N= 3 | scratch    | seed 1 | ±20 0.697 | ±50 0.865 | best thr ±50 0.865
N= 3 | pretrained | seed 2 | ±20 0.737 | ±50 0.947 | best thr ±50 0.947
N= 3 | scratch    | seed 2 | ±20 0.762 | ±50 0.918 | best thr ±50 0.927
N= 9 | pretrained | seed 0 | ±20 0.806 | ±50 0.967 | best thr ±50 0.972
N= 9 | scratch    | seed 0 | ±20 0.780 | ±50 0.941 | best thr ±50 0.941
N= 9 | pretrained | seed 1 | ±20 0.798 | ±50 0.973 | best thr ±50 0.973
N= 9 | scratch    | seed 1 | ±20 0.819 | ±50 0.963 | best t